In [1]:
import time
from datetime import datetime
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import re
import ee

import dask.dataframe as ddf

# from concurrent.futures import ProcessPoolExecutor, as_completed
import os


/home/selker/.conda/envs/geo/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.14) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/selker/.conda/envs/geo/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


# Load data into GDrive using the ee API

In [ ]:
if True:

    today = datetime.now().strftime('%Y-%m-%d')
    ee.Authenticate()
    ee.Initialize(project='eop-alpha-earth-ucb')

    # Get Togo boundary
    togo = ee.FeatureCollection('FAO/GAUL/2015/level0') \
        .filter(ee.Filter.eq('ADM0_NAME', 'Togo'))

    dataset = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')


    # All images are generated in their local Universal Transverse Mercator projection as indicated by the 
    # TM_ZONE property, and have system:time_start and system:time_end properties that reflect the calendar 
    # year summarized by the embeddings; for example, an embedding image for 2021 will have a system:start_time
    # equal to ee.Date('2021-01-01 00:00:00') and a system:end_time equal to ee.Date('2022-01-01 00:00:00').
    # Note: filterDate applies to the start time. Here we use a generous interval to avoid time-zone issues.
    embeddings = dataset.filterDate('2018-12-28', '2019-01-03').mosaic()
    # Clip to Togo
    togo_embeddings = embeddings.clip(togo.geometry())
    # Export in chunks of 16 bands each (64 bands / 4 = 4 files)
    for i in range(4):
        start_band = i * 16
        end_band = (i + 1) * 16
        
        # Select bands start_band through end_band-1
        band_names = [f'A{j:02d}' for j in range(start_band, end_band)]
        subset = togo_embeddings.select(band_names)
        
        task = ee.batch.Export.image.toDrive(
            image=subset,
            description=f'togo_embeddings_bands_{start_band}_{end_band}',
            folder=f'EarthEngineHigherRes_{today}',
            region=togo.geometry(),
            scale=50,
            maxPixels=1e13,
            fileFormat='GeoTIFF'
        )
        task.start()
        print(f'Started export for bands {start_band}-{end_band-1}')

Started export for bands 0-15
Started export for bands 16-31
Started export for bands 32-47
Started export for bands 48-63


# Read in data from TIF, downloaded from gdrive

In [12]:
import pandas as pd
import numpy as np
import rasterio
import rasterio.transform
from pathlib import Path
from pyproj import Transformer
import re

def parse_filename(filename):
    """Extract band range and shard coordinates from filename."""
    # Pattern: prefix_bands_X_Y-SHARD1-SHARD2.tif
    pattern = r'bands_(\d+)_(\d+)-(\d+)-(\d+)\.tif'
    match = re.search(pattern, filename)
    if match:
        return {
            'band_start': int(match.group(1)),
            'band_end': int(match.group(2)),
            'shard_x': int(match.group(3)),
            'shard_y': int(match.group(4))
        }
    return None

def read_tile_to_dataframe(filepath):
    """Read a single GeoTIFF tile and return DataFrame with lat, lon, and band values."""
    with rasterio.open(filepath) as src:
        # Read all bands
        data = src.read()  # Shape: (n_bands, rows, cols)
        n_bands, n_rows, n_cols = data.shape
        
        # Get the transform
        transform = src.transform
        
        # Create coordinate grids for all pixels
        rows, cols = np.meshgrid(np.arange(n_rows), np.arange(n_cols), indexing='ij')
        
        # Convert pixel coordinates to geographic coordinates
        xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
        
        # Flatten the data
        n_pixels = n_rows * n_cols
        
        # Create DataFrame
        df_data = {
            'x': xs,  # Easting in UTM
            'y': ys,  # Northing in UTM
        }
        
        # Add each band as a column
        info = parse_filename(filepath.name)
        assert info is not None
        band_start = info['band_start']
        
        for band_idx in range(n_bands):
            band_num = band_start + band_idx
            df_data[f'band_{band_num}'] = data[band_idx].ravel()
        
        df = pd.DataFrame(df_data)
        
        # Convert UTM to lat/lon
        transformer = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)
        df['lon'], df['lat'] = transformer.transform(df['x'].values, df['y'].values)
        
        # Drop UTM coordinates, keep lat/lon
        df = df.drop(['x', 'y'], axis=1)
        
        return df

def process_tiles_by_shard(directory, pattern='togo_embeddings_bands_*.tif', 
                           output_dir='parquet_output', max_files=None):
    """
    Read all tiles, group by spatial shard, combine bands column-wise,
    and write each shard to a separate parquet file.
    """
    directory = Path(directory)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    files = sorted(directory.glob(pattern))
    if max_files is not None:
        files = files[:max_files]
    
    print(f"Found {len(files)} files")
    
    # Group files by spatial shard (shard_x, shard_y)
    shard_groups = {}
    for filepath in files:
        info = parse_filename(filepath.name)
        assert info is not None
        if info:
            shard_key = (info['shard_x'], info['shard_y'])
            if shard_key not in shard_groups:
                shard_groups[shard_key] = []
            shard_groups[shard_key].append(filepath)
    
    print(f"Found {len(shard_groups)} unique spatial shards")
    
    # Process each shard
    for shard_key, shard_files in shard_groups.items():
        shard_x, shard_y = shard_key
        print(f"\nProcessing shard ({shard_x}, {shard_y}) with {len(shard_files)} files...")
        
        # Read first file to get lat/lon coordinates
        first_df = read_tile_to_dataframe(shard_files[0])
        result_df = first_df[['lat', 'lon']].copy()
        
        # Add bands from all files for this shard
        for filepath in shard_files:
            print(f"  Reading {filepath.name}...")
            df = read_tile_to_dataframe(filepath)
            
            # Get band columns only
            band_cols = [col for col in df.columns if col.startswith('band_')]
            
            # Add band columns to result
            for col in band_cols:
                result_df[col] = df[col].values
        
        # Sort columns: lat, lon, then bands in order
        band_cols = sorted([col for col in result_df.columns if col.startswith('band_')],
                          key=lambda x: int(x.split('_')[1]))
        result_df = result_df[['lat', 'lon'] + band_cols]
        
        # Write to parquet
        output_file = output_dir / f'shard_{shard_x}_{shard_y}.parquet'
        result_df.to_parquet(output_file, index=False)
        
        print(f"  Wrote {output_file} with {len(result_df)} pixels and {len(band_cols)} bands")
    
    print(f"\nDone! All shards written to {output_dir}")
    return output_dir

# Example usage:
# output_dir = process_tiles_by_shard('path/to/tiles', max_files=10)

# To read back with dask:
# import dask.dataframe as dd
# df = dd.read_parquet('parquet_output/*.parquet')

In [13]:
process_tiles_by_shard(
    directory='/data/eop/country_data/TGO/alpha_earth/EarthEngine-2026-01-12', 
    output_dir='/data/eop/country_data/TGO/alpha_earth/EarthEngine-2026-01-12/tabular'
)

Found 8 files
Found 2 unique spatial shards

Processing shard (0, 0) with 4 files...
  Reading togo_embeddings_bands_0_16-0000000000-0000000000.tif...
  Reading togo_embeddings_bands_16_32-0000000000-0000000000.tif...
  Reading togo_embeddings_bands_32_48-0000000000-0000000000.tif...
  Reading togo_embeddings_bands_48_64-0000000000-0000000000.tif...
  Wrote /data/eop/country_data/TGO/alpha_earth/EarthEngine-2026-01-12/tabular/shard_0_0.parquet with 25618688 pixels and 64 bands

Processing shard (5888, 0) with 4 files...
  Reading togo_embeddings_bands_0_16-0000005888-0000000000.tif...
  Reading togo_embeddings_bands_16_32-0000005888-0000000000.tif...
  Reading togo_embeddings_bands_32_48-0000005888-0000000000.tif...
  Reading togo_embeddings_bands_48_64-0000005888-0000000000.tif...
  Wrote /data/eop/country_data/TGO/alpha_earth/EarthEngine-2026-01-12/tabular/shard_5888_0.parquet with 23077704 pixels and 64 bands

Done! All shards written to /data/eop/country_data/TGO/alpha_earth/EarthE

PosixPath('/data/eop/country_data/TGO/alpha_earth/EarthEngine-2026-01-12/tabular')

In [6]:
tabular = ddf.read_parquet(
    '/data/eop/country_data/TGO/alpha_earth/EarthEngine-2026-01-12/tabular/*.parquet'
)

In [13]:
# Drop rows where all band columns are NaN
band_cols = [col for col in tabular.columns if col.startswith('band_')]
tabular_computed = tabular[~tabular[band_cols].isna().all(axis=1)]

In [14]:
# Write
tabular_computed.to_parquet(
    '/data/eop/country_data/TGO/alpha_earth/EarthEngine-2026-01-12/tabular_no_nans',
    write_index=False
)